In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"]='4'
os.environ['HDF5_USE_FILE_LOCKING'] = 'FALSE'
import h5py
import pandas as pd
import numpy as np
import scipy.signal as signal
import scipy.interpolate as interp
from tqdm import tqdm

In [3]:
# --- 2. GT 신호 필터링용 함수 (기존 코드 활용) ---
def butter_bandpass_filter(data, lowcut, highcut, fs, order=2):
    nyq = 0.5 * fs
    low = lowcut / nyq
    high = highcut / nyq
    b, a = signal.butter(order, [low, high], btype='band')
    y = signal.filtfilt(b, a, data)
    return y

def z_norm(data):
    return (data - np.mean(data)) / (np.std(data) + 1e-8)

# --- 3. 핵심 윈도우 처리 및 보간 함수 ---
def process_video_and_gt_windows(video_data, gt_ppg, sub_valid, window_size=900, max_gap_threshold=30):
    n_samples = len(video_data)
    n_windows = n_samples // window_size
    
    valid_vid_list = []
    valid_gt_list = []
    discarded_indices = []

    for i in range(n_windows):
        start_idx = i * window_size
        end_idx = start_idx + window_size
        
        win_video = video_data[start_idx:end_idx].copy()
        win_gt = gt_ppg[start_idx:end_idx].copy()
        win_valid = sub_valid[start_idx:end_idx].copy()
        
        # 결손(Gap) 길이 체크
        is_invalid = ~win_valid
        if np.any(is_invalid):
            condition = np.concatenate(([0], is_invalid, [0]))
            idx = np.flatnonzero(condition[1:] != condition[:-1])
            gap_lengths = idx[1::2] - idx[::2]
            max_gap = gap_lengths.max() if len(gap_lengths) > 0 else 0
        else:
            max_gap = 0
            
        # 임계치 30프레임(1초) 이상 시 폐기
        if max_gap >= max_gap_threshold:
            discarded_indices.append(i)
            continue
            
        try:
            # 보간(Interpolation) 수행
            if max_gap > 0:
                valid_idx = np.where(win_valid)[0]
                all_idx = np.arange(window_size)
                
                if len(valid_idx) > 1:
                    # 다차원 영상 PCHIP 보간
                    win_video = interp.pchip_interpolate(valid_idx, win_video[valid_idx], all_idx, axis=0)
                    

                else:
                    discarded_indices.append(i)
                    continue
            
            # --- 영상 전처리 ---
            # 영상은 주파수 필터링 없이 단순 정규화 (0~1)
            win_video_norm = win_video.astype(np.float32) / 255.0
            
            # --- GT 전처리 ---
            # GT는 Detrend, Bandpass(0.75~2.5Hz), Z-norm 빡빡하게 적용
            detrend_gt = signal.detrend(win_gt)
            filter_gt = butter_bandpass_filter(detrend_gt, 0.75, 2.5, fs_video, order=2)
            win_gt_norm = z_norm(filter_gt)
            
            # 유효 리스트에 추가
            valid_vid_list.append(win_video_norm)
            valid_gt_list.append(win_gt_norm)
            
        except Exception as e:
            print(f'Window {i} Error : {e}')
            discarded_indices.append(i)

    return valid_vid_list, valid_gt_list, discarded_indices

In [9]:
shift_frames = 400
fs_video = 30.0
fs_ppg = 500.0

# --- 1. 설정 및 파라미터 ---
people = ['A001_20250609', 'A002_20250609', 'A004_20250610', 'A005_20250611', 'A006_20250611', 
          'A007_20250612', 'A008_20250612', 'B001_20250623', 'B002_20250623', 'B003_20250624', 
          'B004_20250624', 'B005_20250625', 'B008_20250627', 'C001_20250701', 'C002_20250703', 'C003_20250702']

people_ppg = sorted([k.replace('_20', '_') for k in people])

preprocess_result_dir = '/nasdata4/2yhy/rPPG_code/rPPG_datasets/raw_data/CNA_members/KM_preprocess_data/preprocess_result'
ppg_cut_dir = '/nasdata4/2yhy/rPPG_code/rPPG_datasets/raw_data/CNA_members/KM_preprocess_data/ppg_cut_final' # 실제 GT 경로로 수정 필요
final_save_dir = '/nasdata4/2yhy/rPPG_code/rPPG_datasets/raw_data/CNA_members/KM_preprocess_data/data_for_dl' # 최종 저장 폴더 (미리 생성 필요)

os.makedirs(final_save_dir, exist_ok=True)



# --- 4. 메인 파이프라인 ---
for subject_id, ppg_id in zip(people, people_ppg):
    print(f"\n--- Processing {subject_id} ---")
    
    try:
        # 1. 이전 단계에서 만든 H5 비디오 데이터 로드
        h5_video_path = os.path.join(preprocess_result_dir, f'registration_video_[{subject_id}].h5')
        with h5py.File(h5_video_path, 'r') as f_v:
            video_data = f_v['nir_uv_data'][:]
            sub_valid = f_v['valid_mask'][:]
            
        # 2. GT PPG CSV 로드 (이 경로는 실제 환경에 맞춰 수정해 주세요)
        gt_csv_path = os.path.join(ppg_cut_dir, f'{ppg_id}.csv')
        ppg_cut = pd.read_csv(gt_csv_path).values.flatten()
        
        # 3. 싱크 조절 (400프레임 지연 반영)
        ppg_shift_samples = int(round((shift_frames / fs_video) * fs_ppg))
        
        ppg_trimmed = ppg_cut[ppg_shift_samples:]
        video_trimmed = video_data[:-shift_frames]
        sub_valid_trimmed = sub_valid[:-shift_frames]
        
        # GT 다운샘플링
        ppg_downsampled = signal.resample(ppg_trimmed, len(video_trimmed))
        
        # 4. 윈도우 분할 및 보간
        valid_vid_list, valid_gt_list, discarded = process_video_and_gt_windows(
            video_data=video_trimmed,
            gt_ppg=ppg_downsampled,
            sub_valid=sub_valid_trimmed,
            window_size=900,
            max_gap_threshold=30
        )
        
        if len(valid_vid_list) == 0:
            print(f"[{subject_id}] 유효한 윈도우가 없습니다. 건너뜁니다.")
            continue
            
        # 5. Array 형태로 묶기 (DataLoader 최적화)
        video_array = np.stack(valid_vid_list) # Shape: (유효 윈도우 수, 900, 64, 64)
        gt_array = np.stack(valid_gt_list)     # Shape: (유효 윈도우 수, 900)
        
        # 6. 최종 H5 파일 저장
        final_h5_path = os.path.join(final_save_dir, f'final_dataset_{subject_id}.h5')
        with h5py.File(final_h5_path, 'w') as f_out:
            f_out.create_dataset('video', data=video_array, chunks=True, compression='gzip', compression_opts=4)
            f_out.create_dataset('gt_ppg', data=gt_array, chunks=True, compression='gzip', compression_opts=4)
            
        print(f"[{subject_id}] 완료! 총 {len(valid_vid_list)}개 윈도우 저장됨. (폐기: {len(discarded)}개)")
        print(f"  -> Video Shape: {video_array.shape}, GT Shape: {gt_array.shape}")
        
    except Exception as e:
        print(f"[{subject_id}] 처리 중 오류 발생: {e}")


--- Processing A001_20250609 ---
[A001_20250609] 완료! 총 128개 윈도우 저장됨. (폐기: 1개)
  -> Video Shape: (128, 900, 64, 64), GT Shape: (128, 900)

--- Processing A002_20250609 ---
[A002_20250609] 완료! 총 130개 윈도우 저장됨. (폐기: 6개)
  -> Video Shape: (130, 900, 64, 64), GT Shape: (130, 900)

--- Processing A004_20250610 ---
[A004_20250610] 완료! 총 135개 윈도우 저장됨. (폐기: 1개)
  -> Video Shape: (135, 900, 64, 64), GT Shape: (135, 900)

--- Processing A005_20250611 ---
[A005_20250611] 완료! 총 85개 윈도우 저장됨. (폐기: 58개)
  -> Video Shape: (85, 900, 64, 64), GT Shape: (85, 900)

--- Processing A006_20250611 ---
[A006_20250611] 완료! 총 125개 윈도우 저장됨. (폐기: 0개)
  -> Video Shape: (125, 900, 64, 64), GT Shape: (125, 900)

--- Processing A007_20250612 ---
[A007_20250612] 완료! 총 88개 윈도우 저장됨. (폐기: 55개)
  -> Video Shape: (88, 900, 64, 64), GT Shape: (88, 900)

--- Processing A008_20250612 ---
[A008_20250612] 완료! 총 76개 윈도우 저장됨. (폐기: 63개)
  -> Video Shape: (76, 900, 64, 64), GT Shape: (76, 900)

--- Processing B001_20250623 ---
[B001_

In [14]:
h5_video_path22 = os.path.join(final_save_dir, 'final_dataset_A001_20250609.h5')
with h5py.File(h5_video_path22, 'r') as f_v:
    video_data = f_v['video']
    sub_valid = f_v['gt_ppg']
    print(video_data.shape , sub_valid.shape)
  

(128, 900, 64, 64) (128, 900)
